# 🥉 Bronze Layer: Ingest Raw AAPL Stock Data
**Purpose:** Fetch daily news data from the Alpha Vantage API.

In [0]:
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
from pyspark.sql import functions as F

In [0]:
with open("secrets.json", "r") as file:
        secrets = json.load(file)
        API_KEY = secrets.get("alpha_vantage")

In [0]:
# 1. Configuration
CATALOG = "portfolio"
SCHEMA = "market_data"
BRONZE_NEWS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_news_sentiment"
TICKER = "AAPL"

# Create Catalog, Database if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
# 2. Fetch Data
## From Alpha Vantage (last 24 hours)
time_24h_ago = datetime.now() - timedelta(hours=24)
formatted_time_from = time_24h_ago.strftime("%Y%m%dT%H%M")
url = (
    f"https://www.alphavantage.co/query?"
    f"function=NEWS_SENTIMENT&"
    f"tickers={TICKER}&"
    f"time_from={formatted_time_from}&"
    f"sort=LATEST&"
    f"limit=100&"
    f"apikey={API_KEY}"
)

response = requests.get(url)
data = response.json()

# The actual news articles are inside the 'feed' key
news_feed = data.get("feed", [])

In [0]:
news_feed

In [0]:
# 3. Convert to Spark and Add Metadata
# Alpha Vantage (JSON objects)
if not news_feed:
    print("No news found in the last 24 hours or API limit reached.")
else:
    # 5. Convert to Spark DataFrame and Add Metadata
    df_bronze_news = spark.createDataFrame(news_feed)

    df_bronze_news = df_bronze_news.withColumn("ticker_symbol", F.lit(TICKER)) \
                                   .withColumn("ingestion_timestamp", F.current_timestamp()) \
                                   .withColumn("source_system", F.lit("alphavantage_api"))
                            
    # 4. Write to Bronze (Append)
    df_bronze_news.write.format("delta").mode("append").saveAsTable(BRONZE_NEWS_TABLE)

    display(spark.read.table(BRONZE_NEWS_TABLE).orderBy(F.col("ingestion_timestamp").desc()).limit(5))

In [0]:
if news_feed:
    df_bronze_news.write.format("delta").mode("append").saveAsTable(BRONZE_NEWS_TABLE)

    display(spark.read.table(BRONZE_NEWS_TABLE).orderBy(F.col("ingestion_timestamp").desc()).limit(5))